# 02 — Real-data model training and evaluation

**PathSnap | Official development experiments**

The final model was selected from five classifiers trained using the organiser-provided 80 training trips, with five-fold **GroupKFold** split by trip ID. Ten student-testing trips were used for the reported development comparisons. The ten reserved evaluator trips were not used for fitting, validation, optimisation or selection.

**Selected model: Histogram Gradient Boosting (HGB)** — highest mean training-trip CV Macro F1 (85.90%). The deployed model is the tuned HGB with HMM/Viterbi decoding.

This notebook reads the actual committed experimental CSVs and configurations and links to the production training code. The earlier synthetic-data notebook (including MLP and KNN) is retained under `archive/` for provenance; its results are not comparable with the real-data experiments.

### Recorded baseline results

| Model | CV Macro F1 | Student candidate accuracy | Student Macro F1 | Top-1 proxy accuracy |
|---|---:|---:|---:|---:|
| Logistic Regression | 83.11% | 90.38% | 84.48% | 79.32% |
| Gaussian Naive Bayes | 80.11% | 84.47% | 79.46% | 79.47% |
| Decision Tree | 85.36% | 91.90% | 87.28% | 80.48% |
| Random Forest | 85.63% | 92.09% | 87.36% | 80.74% |
| Histogram Gradient Boosting | 85.90% | 91.94% | 87.22% | 80.47% |

Source: [model_comparison.csv](../results/official_baseline/model_comparison.csv). Candidate accuracy and observation-level Top-1 proxy accuracy are different metrics. Random Forest has slightly higher baseline student-test accuracy, but HGB was selected by training-trip CV Macro F1 and computational efficiency. No MLP was trained on the real dataset.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

cwd = Path.cwd().resolve()
ROOT = next((p for p in (cwd, *cwd.parents) if (p / 'pathsnap_real_training.py').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Open the notebook from inside the PathSnap repository.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print('Repository:', ROOT)

## 1. Actual model configurations and reproducibility

Production implementation: [pathsnap_real_training.py](../pathsnap_real_training.py). Feature preparation uses five nearby road segments per GPS observation and seven numerical features. The reference training labels are nearest-reference-segment proxies, not verified chronological per-fix labels.

The Logistic Regression scaler is fitted inside each fold's pipeline. GroupKFold prevents the same trip appearing in training and validation folds. The primary selection metric is macro-averaged F1, which gives equal weight to each candidate class.

The following cells read saved configurations and comparisons without retraining or replacing model artifacts.

In [ ]:
import json
from pathsnap_real_training import FEATURES, K, SEED, models

BASE = ROOT / 'results' / 'official_baseline'
TUNE = ROOT / 'results' / 'official_tuning'
HMM = ROOT / 'results' / 'official_hmm'
baseline_config = json.loads((ROOT / 'artifacts/official_baseline/configuration.json').read_text())
comparison = pd.read_csv(BASE / 'model_comparison.csv').sort_values('cv_macro_f1', ascending=False)
folds = pd.read_csv(BASE / 'cv_folds.csv')
assert baseline_config['features'] == FEATURES
assert baseline_config['k'] == K and baseline_config['seed'] == SEED
assert set(comparison.model) == set(models()) == set(baseline_config['models'])
assert len(comparison) == 5
assert (folds.groupby('model').size() == 5).all()
for name, model in models().items():
    print(name, ':', model)
print('Features:', FEATURES, '| Candidates:', K, '| Seed:', SEED)

## 2. Trip-grouped validation and student-testing results

These values are read from the actual recorded run. Precision and recall refer to the positive candidate class. CV Macro F1 is the mean of the five training-trip folds; student Macro F1 is measured on the independent student-testing partition. Runtime measurements are for the specific development machine/run.

In [ ]:
display(comparison.reset_index(drop=True))
fold_summary = folds.groupby('model').macro_f1.agg(['mean', 'std', 'min', 'max'])
display(fold_summary)
fold_mean = folds.groupby('model').macro_f1.mean()
assert np.allclose(comparison.cv_macro_f1,
                   comparison.model.map(fold_mean))
best = comparison.iloc[0]
assert best.model == baseline_config['best_by_cv_macro_f1']
print(f'BEST REAL-DATA BASELINE MODEL: {best.model}')
print(f'CV Macro F1: {best.cv_macro_f1:.2%}')
print(f'Student candidate accuracy: {best.candidate_accuracy:.2%}')
print(f'Student Top-1 proxy accuracy: {best.top1_proxy_accuracy:.2%}')

In [ ]:
for metric, title in [
    ('cv_macro_f1', 'Training-trip CV Macro F1'),
    ('student_macro_f1', 'Student-test Macro F1'),
    ('top1_proxy_accuracy', 'Student-test Top-1 proxy accuracy'),
    ('candidate_log_loss', 'Student-test candidate log loss'),
    ('direct_route_continuity', 'Direct route continuity'),
    ('training_seconds', 'Final fit time (seconds)'),
    ('inference_seconds', 'Candidate inference time (seconds)')
]:
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.barh(comparison.model, comparison[metric])
    ax.invert_yaxis()
    ax.set_title(title)
    ax.set_xlabel(metric)
    plt.tight_layout()
    plt.show()

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay
for name in comparison.model:
    slug = name.lower().replace(' ', '_')
    matrix = np.loadtxt(BASE / (slug + '_confusion.csv'), delimiter=',', dtype=int)
    ConfusionMatrixDisplay(matrix, display_labels=[0, 1]).plot(cmap='Blues')
    plt.title('Student-testing candidates — ' + name)
    plt.tight_layout()
    plt.show()

## 3. Bayesian optimisation of selected HGB

[pathsnap_real_tune.py](../pathsnap_real_tune.py) uses Optuna TPE with 15 trials and the same five-fold trip-grouped CV objective. The committed configuration records the best trial and parameters; the full original `bayesian_trials.csv` is not currently committed and must be verified separately. Do not fabricate missing search history.

In [ ]:
tune_config = json.loads((ROOT / 'artifacts/official_tuning/configuration.json').read_text())
tuned = pd.read_csv(TUNE / 'tuned_comparison.csv')
assert tune_config['features'] == FEATURES
assert tune_config['k'] == K
print('Best trial (zero-based):', tune_config['best_trial'])
print('Trial count:', tune_config['trials'])
print('Search time (seconds):', tune_config['search_seconds'])
display(pd.Series(tune_config['best_params'], name='Best value'))
display(tuned[['model','cv_macro_f1','candidate_accuracy','student_macro_f1',
               'top1_proxy_accuracy','candidate_log_loss','inference_seconds']])
history = TUNE / 'bayesian_trials.csv'
if history.is_file():
    display(pd.read_csv(history))
else:
    print('Full trial history is not present. Only recorded configuration and comparison are shown.')

## 4. HMM/Viterbi route reconstruction

The tuned HGB supplies candidate probabilities to the directed Viterbi decoder in [pathsnap_real_hmm.py](../pathsnap_real_hmm.py). The decoder encourages connected transitions; it uses fixed parameters, not a completed decoder optimisation search. The recorded comparisons below are student-testing results, not reserved evaluator results.

In [ ]:
decoder = json.loads((HMM / 'decoder_config.json').read_text())
route_results = pd.read_csv(HMM / 'hmm_comparison.csv')
display(pd.Series(decoder, name='Decoder configuration'))
display(route_results)
metrics = ['top1_proxy_accuracy', 'direct_route_continuity',
           'mean_sequence_overlap_f1', 'mean_ordered_sequence_similarity']
route_results.set_index('method')[metrics].T.plot.bar(figsize=(10, 5), rot=15)
plt.ylim(0, 1)
plt.ylabel('Score (0–1)')
plt.title('Student-testing route reconstruction')
plt.tight_layout()
plt.show()

## 5. Reproduce training (optional; not automatically executed)

The official scripts contain the fitting and evaluation code. To reproduce the experiments locally, first place the organiser's development data in `data/train` and `data/student_test`. Use a **separate working copy** if you need to keep the current frozen model and original results intact. Running these scripts writes new weights and results.

```bash
python -u pathsnap_real_training.py prepare
python -u pathsnap_real_training.py train
python -u pathsnap_real_tune.py
python -u pathsnap_real_hmm.py
```

The notebooks intentionally do not silently train on synthetic data or overwrite the deployed model. The selected model is determined by the recorded real-data CV results, not by a model name hardcoded into the scoring logic.

## Conclusion

The official five-model comparison selects HGB; Bayesian optimisation improved its CV Macro F1 to 86.00%. The tuned HGB + HMM/Viterbi system achieved 84.06% pointwise proxy accuracy and 95.36% direct route continuity on student testing. The separate evaluator API loads the frozen tuned HGB. Do not use held-out evaluator results to modify the model.

Limitations: reference labels are proxies; the decoder does not guarantee a fully connected route or insert intermediate arcs. Full original tuning history and decoder optimisation remain outstanding.